# Day 202 — Streaming, Parallel Nodes & Error Handling/Retries
### Month 13: Agentic AI & Advanced GenAI Portfolio | Week 1, Day 3 (LangGraph)

**Scenario:** TeleServe India support-ticket triage, extended from Day200/Day201.
Instead of one classification model doing everything sequentially, three independent
classifiers (urgency, sentiment, category) run **in parallel** on each ticket, then a
merge node combines them into a final priority. We also make the pipeline
production-honest by adding **retry logic** for transient API failures and
**streaming** so a caller can see progress ticket-by-ticket instead of waiting for
the whole batch.

**Builds on:** Day200 (StateGraph, conditional edges, interrupt), Day201 (multi-turn
state, checkpointing, `add_messages` reducer pattern).

**New concepts today:**
1. Fan-out / fan-in graphs (parallel node execution in a single superstep)
2. `Annotated` reducers as the *mechanism*, not just multi-turn message lists
3. `RetryPolicy` on nodes — and a verified gotcha about which exceptions it retries by default
4. `.stream(..., stream_mode=...)` for incremental output

Five sections, same locked format as every prior day: **Raw Data → Concept Notes →
Practice Guide → Answer Key → Scoring Rubric.**


---
## Section 1: Raw Data (immutable — never modify this cell's output)

10 TeleServe India support tickets, generated with `seed=202` for reproducibility.
Treat this exactly like a locked CSV import — build everything downstream in
separate cells/functions, never edit `RAW_TICKETS` in place.


In [1]:
import random

random.seed(202)

CATEGORIES = ["billing", "network", "device", "account", "roaming"]
TIERS = ["Standard", "Silver", "Gold", "Platinum"]

TEMPLATES = [
    "My bill this month is double what I usually pay and nobody has explained why.",
    "Internet has been down for two days in my area, this is extremely urgent.",
    "The SIM card you sent is not activating, I have tried three times.",
    "I want to close my account, I am switching to another provider.",
    "Roaming charges appeared on my bill even though I disabled roaming before my trip.",
    "Router keeps disconnecting every 10 minutes, very frustrating experience.",
    "Please update my registered address, the form on the website is broken.",
    "I was charged twice for the same recharge plan this week.",
    "Signal strength is extremely poor since the recent tower maintenance.",
    "Can you confirm if my premium plan includes international roaming minutes?",
]

RAW_TICKETS = []
for i, text in enumerate(TEMPLATES, start=1):
    RAW_TICKETS.append({
        "ticket_id": i,
        "customer_tier": random.choice(TIERS),
        "ticket_text": text,
    })

for t in RAW_TICKETS:
    print(t)


{'ticket_id': 1, 'customer_tier': 'Platinum', 'ticket_text': 'My bill this month is double what I usually pay and nobody has explained why.'}
{'ticket_id': 2, 'customer_tier': 'Platinum', 'ticket_text': 'Internet has been down for two days in my area, this is extremely urgent.'}
{'ticket_id': 3, 'customer_tier': 'Platinum', 'ticket_text': 'The SIM card you sent is not activating, I have tried three times.'}
{'ticket_id': 4, 'customer_tier': 'Silver', 'ticket_text': 'I want to close my account, I am switching to another provider.'}
{'ticket_id': 5, 'customer_tier': 'Silver', 'ticket_text': 'Roaming charges appeared on my bill even though I disabled roaming before my trip.'}
{'ticket_id': 6, 'customer_tier': 'Platinum', 'ticket_text': 'Router keeps disconnecting every 10 minutes, very frustrating experience.'}
{'ticket_id': 7, 'customer_tier': 'Standard', 'ticket_text': 'Please update my registered address, the form on the website is broken.'}
{'ticket_id': 8, 'customer_tier': 'Platinum'

---
## Section 2: Concept Notes

### 2.1 Why parallel nodes at all?
On Day200/201, every node ran one after another (sequential supersteps). That's
correct for a conversation, where turn 2 depends on turn 1. But urgency, sentiment,
and category classification for the **same ticket** don't depend on each other —
running them sequentially just adds latency for no accuracy benefit. LangGraph runs
multiple nodes **in the same superstep** whenever they share the same incoming edge
source, which is the fan-out pattern:

```
        ┌─► classify_urgency ───┐
intake ─┼─► classify_sentiment ─┼─► merge_and_prioritize ─► END
        └─► classify_category ──┘
```

`intake` has three outgoing edges. LangGraph schedules all three destination nodes
in the same step (concurrently, via a thread pool internally) and only advances to
`merge_and_prioritize` once **all three** have returned — that's the fan-in / join.

### 2.2 The reducer requirement — verified, not assumed
A natural instinct is to have all three classifiers write to one shared field like
`"log"` or even the same field. Tested directly against a live `langgraph==1.2.8`
install:

```python
class BadState(TypedDict):
    shared_field: Optional[str]

def a(state): return {"shared_field": "from_a"}
def b(state): return {"shared_field": "from_b"}
# both fan out from START, both write shared_field, no reducer
```

Running this raises, verbatim:

```
InvalidUpdateError: At key 'shared_field': Can receive only one value per step.
Use an Annotated key to handle multiple values.
```

**Why this happens:** in a single superstep, LangGraph collects every node's return
dict and merges them into the new state. If two nodes touch the *same key* and that
key has no reducer, the merge is ambiguous — LangGraph refuses to silently pick a
winner and raises instead. This is the parallel-node analogue of Day201's
same-step-staleness lesson: concurrent writes need an explicit merge strategy.

**The fix has two valid shapes:**
- **Different keys, no reducer needed** — `urgency`, `sentiment`, `category` are each
  owned by exactly one node, so plain `Optional[str]` fields are fine.
- **Same key, reducer required** — an append-only trail like `log` that *multiple*
  nodes legitimately write to needs `Annotated[List[str], operator.add]`, exactly
  like Day201's `add_messages` pattern but with the simpler built-in `operator.add`
  reducer for plain lists.

### 2.3 RetryPolicy — and a gotcha worth knowing before it costs you in production
LangGraph 1.2.8's `add_node()` takes a `retry_policy` argument:

```python
from langgraph.types import RetryPolicy
builder.add_node("classify_category", classify_category,
                  retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.5,
                                            backoff_factor=2.0, jitter=True))
```

The instinct is to assume "any exception triggers a retry." Verified against the
live install by reading `langgraph.types.default_retry_on` directly:

```python
def default_retry_on(exc):
    if isinstance(exc, ConnectionError): return True
    if isinstance(exc, httpx.HTTPStatusError): return 500 <= exc.response.status_code < 600
    if isinstance(exc, requests.HTTPError): return 500 <= status < 600 if response else True
    if isinstance(exc, (ValueError, TypeError, ArithmeticError, ImportError,
                         LookupError, NameError, SyntaxError, RuntimeError,
                         ReferenceError, StopIteration, StopAsyncIteration, OSError)):
        return False
    return True
```

**The gotcha:** `ValueError`, `TypeError`, `RuntimeError` and several other common
built-in exceptions are **explicitly excluded** from the default retry list. If a
Groq/httpx wrapper raises a `ValueError` on a malformed response (a very normal
thing for an LLM-parsing layer to do), the node fails **once, immediately, no
retry** — even with `RetryPolicy` attached — because the default `retry_on`
function says "don't retry this." Confirmed empirically: a node raising
`ValueError` propagated straight through `RetryPolicy(max_attempts=4)` on the first
attempt; the identical node raising `ConnectionError` correctly retried and
succeeded on attempt 3.

**Practical rule for this pipeline:** transient Groq API failures surface as
`ConnectionError` or `httpx.HTTPStatusError` (5xx) — both retry by default, so the
out-of-the-box policy is actually correct for real network flakiness. But any
`ValueError` you raise yourself (e.g. "model returned an unparseable label") needs
an explicit `retry_on` if you want it retried, because by default it won't be:

```python
RetryPolicy(max_attempts=3, retry_on=lambda exc: True)   # retry everything
# or, narrower and safer:
RetryPolicy(max_attempts=3, retry_on=lambda exc: isinstance(exc, (ConnectionError, ValueError)))
```

### 2.4 Streaming
`app.invoke()` blocks until the whole graph finishes. `app.stream()` yields a chunk
after every superstep, which matters a lot once nodes run in parallel — a caller
watching the stream sees three classifier outputs land together, then the merge
node's output. Stream modes (verified against `langgraph.types.StreamMode`):
`"values"` (full state after each step), `"updates"` (only what each node changed —
what we use today, since we want to see each classifier's individual contribution),
`"messages"`, `"debug"`, `"checkpoints"`, `"tasks"`, `"custom"`.

### 2.5 Real-world use
A telecom or SaaS support desk processing thousands of tickets/day cannot afford
sequential LLM calls per ticket (3x the latency, 3x the time-to-first-response).
Fan-out classification + retry-hardened nodes + streamed progress is the actual
shape of production ticket-triage pipelines — not a toy simplification.


---
## Section 3: Practice Guide

Only use what Day200–201 already covered plus today's Concept Notes. No new syntax
beyond what's introduced above.

### Setup (run once)


In [2]:
!pip install -q langgraph==1.2.8 langchain-groq==1.1.2 --break-system-packages
# Runtime -> Restart after installing, then re-run from the top.


In [3]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from typing import TypedDict, Annotated, List, Optional
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
print("LLM ready:", llm.model_name)


LLM ready: openai/gpt-oss-20b


### Task 1 — State schema with correct reducer choices (20 pts)
Define `TicketState`. Decide, per field, whether it needs `Annotated[..., operator.add]`
or a plain `Optional[...]`, and be ready to justify each choice against the
`InvalidUpdateError` behavior from Concept Notes 2.2.

- `ticket_id: int`, `ticket_text: str`, `customer_tier: str` — set once by intake, read-only after
- `urgency`, `sentiment`, `category`: `Optional[str]` — each owned by exactly one parallel node
- `log`: written by **every** node (intake + all 3 classifiers + merge) — needs a reducer
- `final_priority: Optional[str]` — set once by the merge node

```python
class TicketState(TypedDict):
    # your fields here
    ...
```


In [4]:
# ── TASK 1 (20 pts): Define the state schema ─────────────────────────────
# Goal: Create a TypedDict with appropriate reducers for concurrent writes.
# Method:
#   - Fields that are written by exactly one node: plain Optional[str]
#   - Fields written by multiple nodes (log): use Annotated with operator.add
#   - Justification: Avoiding InvalidUpdateError when multiple nodes in same
#     superstep write to the same key; log needs aggregation, others are owned.

from typing import TypedDict, Annotated, List, Optional
import operator

class TicketState(TypedDict):
    # Read-only fields set by intake
    ticket_id: int
    ticket_text: str
    customer_tier: str

    # Each owned by exactly one classifier – no reducer needed
    urgency: Optional[str]        # set by classify_urgency
    sentiment: Optional[str]      # set by classify_sentiment
    category: Optional[str]       # set by classify_category

    # Written by every node – needs aggregation
    log: Annotated[List[str], operator.add]

    # Set once by merge node – no reducer needed
    final_priority: Optional[str]

### Task 2 — Three parallel LLM-calling classifier nodes (25 pts)
Each node:
- Reads `ticket_text` from state
- Calls `llm.invoke(...)` with a prompt that forces a single-word/short-label answer
  (temperature=0 already set on the shared `llm` above — do not create new LLM
  instances per node)
- Returns **only its own field** + appends one string to `log`
- `classify_urgency` → `"high"` or `"normal"`
- `classify_sentiment` → `"positive"`, `"neutral"`, or `"negative"`
- `classify_category` → one of `CATEGORIES` from Raw Data

```python
def classify_urgency(state: TicketState) -> dict:
    ...

def classify_sentiment(state: TicketState) -> dict:
    ...

def classify_category(state: TicketState) -> dict:
    ...
```


In [5]:
# ── TASK 2 (25 pts): Three parallel LLM-calling classifier nodes ────────
# Goal: Each node calls the shared LLM to classify one aspect of the ticket.
# Method:
#   - Each node reads ticket_text from state.
#   - Uses a prompt to force a single-word/short answer.
#   - Returns only its own classification field + appends one log entry.
#   - Does NOT create a new LLM instance – uses the shared `llm`.

def classify_urgency(state: TicketState) -> dict:
    """Classify urgency as 'high' or 'normal'."""
    prompt = (
        f"Ticket text: {state['ticket_text']}\n"
        "Classify the urgency of this ticket as exactly 'high' or 'normal'. "
        "Answer with only one word."
    )
    response = llm.invoke(prompt)
    resp = response.content.strip().lower()
    log_entries = []

    # Substring matching, not exact equality
    if "high" in resp:
        urgency = "high"
    elif "normal" in resp:
        urgency = "normal"
    else:
        urgency = "normal"
        log_entries.append(f"WARNING: urgency fallback used for response: '{resp}'")

    log_entries.append(f"urgency_classifier: {urgency}")
    return {"urgency": urgency, "log": log_entries}

def classify_sentiment(state: TicketState) -> dict:
    """Classify sentiment as 'positive', 'neutral', or 'negative'."""
    prompt = (
        f"Ticket text: {state['ticket_text']}\n"
        "Classify the sentiment of this ticket as exactly 'positive', 'neutral', or 'negative'. "
        "Answer with only one word."
    )
    response = llm.invoke(prompt)
    resp = response.content.strip().lower()
    log_entries = []

    if "positive" in resp:
        sentiment = "positive"
    elif "neutral" in resp:
        sentiment = "neutral"
    elif "negative" in resp:
        sentiment = "negative"
    else:
        sentiment = "neutral"
        log_entries.append(f"WARNING: sentiment fallback used for response: '{resp}'")

    log_entries.append(f"sentiment_classifier: {sentiment}")
    return {"sentiment": sentiment, "log": log_entries}

def classify_category(state: TicketState) -> dict:
    """Classify category as one of the CATEGORIES list."""
    prompt = (
        f"Ticket text: {state['ticket_text']}\n"
        f"Classify the category of this ticket as exactly one of: {', '.join(CATEGORIES)}. "
        "Answer with only one word."
    )
    response = llm.invoke(prompt)
    resp = response.content.strip().lower()
    log_entries = []

    # Check if response contains any valid category
    matched = next((c for c in CATEGORIES if c in resp), None)
    if matched:
        category = matched
    else:
        category = "billing"  # fallback
        log_entries.append(f"WARNING: category fallback used for response: '{resp}'")

    log_entries.append(f"category_classifier: {category}")
    return {"category": category, "log": log_entries}


### Task 3 — Wire the fan-out / fan-in graph + merge logic (20 pts)
- `intake` node: just logs that intake ran (no LLM call, no classification — that's
  what makes it a clean fan-out point)
- Three edges from `intake` to the three classifiers (fan-out)
- Three edges from the three classifiers into `merge_and_prioritize` (fan-in)
- `merge_and_prioritize` computes `final_priority` with a **Number + Reason + Action**
  style rule using the three classified fields (e.g. urgency=high AND
  sentiment=negative → `"P1"`; category=network AND urgency=high → `"P1"`; else
  tiered by `customer_tier`; else `"P3"`) — write the rule yourself, don't just
  print a placeholder
- Compile and run `app.get_graph().print_ascii()` (or `.draw_mermaid()` if ascii
  isn't available) to visually confirm the fan-out/fan-in shape before moving on

```python
builder = StateGraph(TicketState)
# add_node calls
# add_edge calls (fan-out then fan-in)
app = builder.compile()
```


In [6]:
# ── TASK 3 (20 pts): Wire the fan-out/fan-in graph + merge logic ────────
# Goal: Build the graph with intake fanning out to three classifiers,
#       then fanning in to a merge node that computes final_priority.
# Method:
#   - intake node: just logs that the pipeline started.
#   - Three edges from intake to each classifier (fan-out).
#   - Three edges from classifiers to merge (fan-in).
#   - merge node: applies a deterministic business rule to set priority.
#   - Print the graph structure for visual verification.

from langgraph.graph import StateGraph, START, END

def intake_node(state: TicketState) -> dict:
    """Initial node – just logs the start of processing."""
    return {"log": [f"intake: processing ticket {state['ticket_id']}"]}

def merge_and_prioritize(state: TicketState) -> dict:
    """
    Compute final_priority using a deterministic rule based on classified fields.
    Rule:
      - urgency=high AND sentiment=negative -> P1
      - urgency=high AND category=network -> P1
      - customer_tier=Platinum -> P2
      - else -> P3
    """
    urgency = state.get("urgency")
    sentiment = state.get("sentiment")
    category = state.get("category")
    tier = state.get("customer_tier")

    if urgency == "high" and sentiment == "negative":
        priority = "P1"
    elif urgency == "high" and category == "network":
        priority = "P1"
    elif tier == "Platinum":
        priority = "P2"
    else:
        priority = "P3"

    return {
        "final_priority": priority,
        "log": [f"merge: assigned priority {priority}"]
    }

# Build the graph
builder = StateGraph(TicketState)

# Add nodes
builder.add_node("intake", intake_node)
builder.add_node("classify_urgency", classify_urgency)
builder.add_node("classify_sentiment", classify_sentiment)
builder.add_node("classify_category", classify_category)
builder.add_node("merge", merge_and_prioritize)

# Fan-out: from intake to all three classifiers
builder.add_edge(START, "intake")
builder.add_edge("intake", "classify_urgency")
builder.add_edge("intake", "classify_sentiment")
builder.add_edge("intake", "classify_category")

# Fan-in: from classifiers to merge
builder.add_edge("classify_urgency", "merge")
builder.add_edge("classify_sentiment", "merge")
builder.add_edge("classify_category", "merge")

# Merge to END
builder.add_edge("merge", END)

# Compile (without checkpointer for now – retry and streaming later)
app_no_retry = builder.compile()

# Visualise the graph (ASCII fallback if mermaid not available)
try:
    print(app_no_retry.get_graph().draw_mermaid())
except:
    app_no_retry.get_graph().print_ascii()

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	classify_urgency(classify_urgency)
	classify_sentiment(classify_sentiment)
	classify_category(classify_category)
	merge(merge)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	classify_category --> merge;
	classify_sentiment --> merge;
	classify_urgency --> merge;
	intake --> classify_category;
	intake --> classify_sentiment;
	intake --> classify_urgency;
	merge --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Task 4 — RetryPolicy: production policy + isolated proof (20 pts)

**4a — Production policy.** Attach a `RetryPolicy` to all three LLM-calling nodes.
Since Groq/httpx transient failures surface as `ConnectionError` or 5xx
`httpx.HTTPStatusError` (both retried by default per Concept Notes 2.3), the default
`retry_on` is actually correct here — but set `max_attempts=3` explicitly rather than
relying on the library default, and state in a comment why you are or aren't
overriding `retry_on`.

**4b — Isolated proof the retry mechanism actually works.** Build one small
standalone node with no LLM call — `flaky_demo_node` — that deterministically raises
`ConnectionError` on its first 2 calls (use a module-level counter dict, not a class
attribute) and succeeds on the 3rd, returning a distinguishable log message that
includes the attempt number. Attach `RetryPolicy(max_attempts=4, initial_interval=0.05,
backoff_factor=1.0, jitter=False)`. Run this node through its own tiny single-node
graph (`invoke()`), and print the counter afterward to prove it took exactly 3
attempts — this is your Number for today's NRA-style verification, not an assumption.

```python
_attempt_counter = {"n": 0}

def flaky_demo_node(state) -> dict:
    ...

# separate tiny graph just for this node, invoke it, print the counter
```


In [7]:
# ── TASK 4a (part of 20 pts): RetryPolicy on three real classifier nodes ──
# Goal: Attach RetryPolicy to each LLM‑calling node to handle transient API failures.
# Method:
#   - Use RetryPolicy(max_attempts=3, initial_interval=0.5, backoff_factor=2.0, jitter=True).
#   - Keep default retry_on because Groq errors (ConnectionError, 5xx) are retried.
#   - Rebuild the graph with retry_policy added.
#   - State in comment why default retry_on is sufficient.

from langgraph.types import RetryPolicy

# Retry policy for all LLM nodes
# Default retry_on catches ConnectionError and HTTP 5xx, which are the typical
# transient failures from Groq/httpx. ValueError etc. are not retried, but
# our nodes only raise such errors if parsing fails – we handle that with fallbacks,
# so we don't need to override retry_on.
RETRY_POLICY = RetryPolicy(
    max_attempts=3,
    initial_interval=0.5,
    backoff_factor=2.0,
    jitter=True
)

# Rebuild the graph with retry_policy
builder_retry = StateGraph(TicketState)
builder_retry.add_node("intake", intake_node)
builder_retry.add_node("classify_urgency", classify_urgency, retry_policy=RETRY_POLICY)
builder_retry.add_node("classify_sentiment", classify_sentiment, retry_policy=RETRY_POLICY)
builder_retry.add_node("classify_category", classify_category, retry_policy=RETRY_POLICY)
builder_retry.add_node("merge", merge_and_prioritize)

builder_retry.add_edge(START, "intake")
builder_retry.add_edge("intake", "classify_urgency")
builder_retry.add_edge("intake", "classify_sentiment")
builder_retry.add_edge("intake", "classify_category")
builder_retry.add_edge("classify_urgency", "merge")
builder_retry.add_edge("classify_sentiment", "merge")
builder_retry.add_edge("classify_category", "merge")
builder_retry.add_edge("merge", END)

app = builder_retry.compile()
print("Graph compiled with RetryPolicy on classifier nodes.")

Graph compiled with RetryPolicy on classifier nodes.


In [8]:
# ── TASK 4b (part of 20 pts): Isolated proof that retry mechanism works ──
# Goal: Build a standalone flaky node that deterministically fails first 2 times,
#       then succeeds on the 3rd, and prove it via printed counter.
# Method:
#   - Use a module-level counter dict to track attempts.
#   - flaky_demo_node raises ConnectionError on attempts 1 and 2, returns on 3.
#   - Build a tiny graph with just that node and RetryPolicy, invoke it.
#   - Print the counter after invocation to show exactly 3 attempts.

from typing import TypedDict, Annotated, List
import operator

class FlakyState(TypedDict):
    log: Annotated[List[str], operator.add]   # exercises reducer even with single node

_attempt_counter = {"n": 0}

def flaky_demo_node(state: FlakyState) -> dict:
    _attempt_counter["n"] += 1
    attempt = _attempt_counter["n"]
    if attempt <= 2:
        raise ConnectionError(f"Simulated transient failure on attempt {attempt}")
    return {"log": [f"flaky_demo succeeded on attempt {attempt}"]}

flaky_builder = StateGraph(FlakyState)
flaky_builder.add_node("flaky", flaky_demo_node, retry_policy=RetryPolicy(
    max_attempts=4,
    initial_interval=0.05,
    backoff_factor=1.0,
    jitter=False
))
flaky_builder.add_edge(START, "flaky")
flaky_builder.add_edge("flaky", END)
flaky_app = flaky_builder.compile()

result = flaky_app.invoke({"log": []})
print("flaky_demo_node succeeded. Final state:", result)
print(f"Total attempts recorded: {_attempt_counter['n']}")
assert _attempt_counter["n"] == 3
print("✅ Verification: flaky node took exactly 3 attempts.")

flaky_demo_node succeeded. Final state: {'log': ['flaky_demo succeeded on attempt 3']}
Total attempts recorded: 3
✅ Verification: flaky node took exactly 3 attempts.


### Task 5 — Stream the full batch + execution proof (15 pts)
For all 10 tickets in `RAW_TICKETS`:
- Run each through `app.stream(initial_state, stream_mode="updates")`
- Print each streamed chunk as it arrives (this is your visible proof that classifier
  nodes are landing as separate chunks within the same step, not one giant blocking
  wait)
- After the stream finishes for each ticket, print the ticket_id and `final_priority`
- At the end, print a one-line summary table: ticket_id → final_priority, sorted so
  all `"P1"` tickets are shown first

Do **not** use `app.invoke()` for this task — the point is to prove streaming
behavior, and a printed transcript from `.stream()` is the only acceptable evidence.

```python
initial_state = {
    "ticket_id": t["ticket_id"],
    "ticket_text": t["ticket_text"],
    "customer_tier": t["customer_tier"],
    "urgency": None, "sentiment": None, "category": None,
    "log": [], "final_priority": None,
}
for chunk in app.stream(initial_state, stream_mode="updates"):
    print(chunk)
```


In [9]:
# ── TASK 5 (15 pts): Stream the full batch + execution proof ─────────────
# Goal: Process all 10 tickets using stream_mode="updates" to show per‑chunk
#       progress, then print a sorted summary of final priorities.
# Method:
#   - For each ticket, build initial state and call app.stream(…).
#   - Print every chunk as it arrives to prove parallel streaming.
#   - Capture the merge node's update, which contains final_priority.
#   - After all tickets, print a summary table sorted P1 → P2 → P3.

print("\n--- Streaming batch processing ---\n")
summary = []

for ticket in RAW_TICKETS:
    ticket_id = ticket["ticket_id"]
    print(f"\n=== Processing ticket {ticket_id} ===")

    initial_state = {
        "ticket_id": ticket_id,
        "ticket_text": ticket["ticket_text"],
        "customer_tier": ticket["customer_tier"],
        "urgency": None,
        "sentiment": None,
        "category": None,
        "log": [],
        "final_priority": None,
    }

    # Hold the merge update so we can extract priority after streaming.
    final_state = None

    # Stream updates; each chunk is a dict of {node_name: updated_values}.
    for chunk in app.stream(initial_state, stream_mode="updates"):
        print(f"  {chunk}")
        # The merge node's update contains the final priority.
        if "merge" in chunk:
            final_state = chunk["merge"]

    # Extract priority from the captured merge update.
    priority = final_state.get("final_priority", "UNKNOWN") if final_state else "UNKNOWN"
    summary.append((ticket_id, priority))
    print(f"  -> Final priority for ticket {ticket_id}: {priority}")

# Print summary table, sorted alphabetically (P1 < P2 < P3).
print("\n--- Summary: ticket_id -> final_priority (P1 first) ---")
sorted_summary = sorted(summary, key=lambda x: x[1])
for ticket_id, priority in sorted_summary:
    print(f"  Ticket {ticket_id}: {priority}")


--- Streaming batch processing ---


=== Processing ticket 1 ===
  {'intake': {'log': ['intake: processing ticket 1']}}
  {'classify_sentiment': {'sentiment': 'negative', 'log': ['sentiment_classifier: negative']}}
  {'classify_category': {'category': 'billing', 'log': ['category_classifier: billing']}}
  {'classify_urgency': {'urgency': 'high', 'log': ['urgency_classifier: high']}}
  {'merge': {'final_priority': 'P1', 'log': ['merge: assigned priority P1']}}
  -> Final priority for ticket 1: P1

=== Processing ticket 2 ===
  {'intake': {'log': ['intake: processing ticket 2']}}
  {'classify_category': {'category': 'network', 'log': ['category_classifier: network']}}
  {'classify_sentiment': {'sentiment': 'negative', 'log': ['sentiment_classifier: negative']}}
  {'classify_urgency': {'urgency': 'high', 'log': ['urgency_classifier: high']}}
  {'merge': {'final_priority': 'P1', 'log': ['merge: assigned priority P1']}}
  -> Final priority for ticket 2: P1

=== Processing ticket 3 ===
  {'

---
## Section 4: Scoring Rubric — 100 points

| Task | Criteria | Points |
|---|---|---|
| Task 1 — State schema | `Annotated[List[str], operator.add]` used only for `log`; `urgency`/`sentiment`/`category`/`final_priority` are plain `Optional[str]`; can justify choice against `InvalidUpdateError` | 20 |
| Task 2 — Parallel classifier nodes | 3 nodes, each calls the shared `llm` (no new LLM instances), each returns only its own field + one log entry, no cross-field writes | 25 |
| Task 3 — Fan-out/fan-in wiring + merge rule | Correct edge topology (3-out from intake, 3-in to merge); `final_priority` rule is a real deterministic business rule (not a placeholder); graph printed/visualized | 20 |
| Task 4 — RetryPolicy (4a + 4b) | 4a: retry_policy attached to all 3 real nodes with explicit max_attempts + stated reasoning on retry_on; 4b: isolated flaky node proves exactly 3 attempts via printed counter, not assumed | 20 |
| Task 5 — Streaming + execution proof | Uses `.stream(..., stream_mode="updates")` not `.invoke()`; per-ticket chunks printed; final_priority printed per ticket; sorted summary table at the end | 15 |
| **Total** | | **100** |

**Technical vs. Communication split (per Universal Project Instructions #6):**
- *Technical failure example:* using `operator.add` reducer on `urgency` (wrong — only
  one node ever writes it, and using a reducer there hides a real bug if two nodes
  ever did collide) — wrong logic
- *Communication failure example:* correct code, but the Task 4a comment doesn't
  explain *why* the default `retry_on` is or isn't sufficient for this pipeline —
  right answer, unexplained reasoning, fails the "explain it to a client" bar

### Interview-framing answer (Universal Instruction #11)
*"How would you explain today's work to a client or interviewer?"*

> "I moved ticket classification from three sequential LLM calls to three parallel
> ones that run in the same step, which cuts triage latency roughly 3x since urgency,
> sentiment, and category don't depend on each other. I also hardened the pipeline
> against real API flakiness with a retry policy — but I didn't just add it and
> assume it worked. I checked the library's default retry logic directly and found
> it silently skips retrying certain exception types like ValueError, so I verified
> with an isolated test that my retry policy actually fires before trusting it in
> the real pipeline. That's the difference between code that looks resilient and
> code that's proven resilient."

### GitHub note (Universal Instruction #12)
This extends the same `Month13-AgenticAI-Portfolio` repo as Day200/201 — no new repo
yet. Fan-out/fan-in + verified-retry pattern is a strong candidate snippet to
highlight in that repo's README once Week 4's capstone repo is live.
